# La struttura degli argomenti — la nostra ipotesi, marcata sui dati

Nel notebook 03 i temi *emergevano* dai dati e confermavano una mappa. Qui
facciamo il passo dichiarato: **strutturiamo la nostra ipotesi** di argomenti in
**famiglie** e con quelle **marchiamo** i testi. Le famiglie sono il *telaio* —
non qualcosa da sottrarre, ma la griglia con cui leggere di cosa parlano i Papi.

Due scelte di metodo importanti:
- **Si marca per significato, non per parole.** Ogni famiglia ha un'*ancora*
  semantica (una frase che la descrive); un pezzo di testo prende la famiglia la
  cui ancora gli è più vicina (coseno). Niente soglie: si sceglie la più vicina
  fra le sei (vedi notebook 02 sul perché).
- **I discorsi si mischiano.** Un'omelia può contenere liturgia *e* attualità: per
  questo marchiamo i **chunk**, non i documenti interi. Un documento risulta così
  un **mix** di famiglie — che è la verità dei testi.

## Le sei famiglie (la nostra ipotesi, strutturata)

| # | famiglia | cosa raccoglie |
|---|---|---|
| 1 | **liturgia** | la Messa, i sacramenti, l'anno liturgico (Avvento, Natale, Quaresima, Pasqua, Pentecoste) |
| 2 | **fede e devozione** | Dio, Gesù, lo Spirito Santo, la Vergine Maria e i santi, la preghiera, il Vangelo — la *lunga linea rossa* |
| 3 | **eventi e ricorrenze** | udienze a gruppi e categorie, anniversari e giubilei, i corpi e le accademie, le visite *ad limina* dei vescovi — ciò che gli viene **organizzato** |
| 4 | **viaggi** | i viaggi apostolici e le visite pastorali, i Paesi e i popoli incontrati |
| 5 | **programma e storia** | la linea che ogni Papa imprime: dottrina sociale, ecumenismo e dialogo, il rapporto coi grandi eventi storici |
| 6 | **attualità** | pace e guerre, migranti, ambiente e clima, economia e lavoro, tecnologia e intelligenza artificiale |

## Come gira

Una sola passata: embeddo le sei ancore col modello, poi per ogni chunk
dell'indice prendo l'ancora più vicina. Il prodotto è in **torch** (vedi nota
ambiente nel README di `analisi/`).

In [ ]:
import collections
import numpy as np
import torch
from ponte import embedder, tabella

FAM = {
 "liturgia": "la liturgia e i sacramenti: la santa Messa, l'Eucaristia, l'anno liturgico, l'Avvento, il Natale, la Quaresima, la Pasqua, la Pentecoste",
 "fede e devozione": "Dio, Gesù Cristo, lo Spirito Santo, la Vergine Maria e i santi, la preghiera, il Vangelo, la fede e la salvezza",
 "eventi e ricorrenze": "udienze e incontri con gruppi, categorie, istituzioni; ricorrenze, anniversari, giubilei; i corpi e le accademie; le visite ad limina dei vescovi",
 "viaggi": "i viaggi apostolici e le visite pastorali, l'arrivo e il saluto nei Paesi visitati, le nazioni e i popoli incontrati lungo il cammino",
 "programma e storia": "il programma del pontificato e il suo segno nella storia: la dottrina sociale, l'ecumenismo e il dialogo, il rapporto coi grandi eventi storici del tempo",
 "attualità": "i temi dell'attualità: la pace e le guerre, i migranti e i rifugiati, l'ambiente e il clima, l'economia e il lavoro, la tecnologia e l'intelligenza artificiale",
}
FN = list(FAM)
emb = embedder()
A = torch.from_numpy(np.stack([emb.query(v) for v in FAM.values()]).astype("float32"))  # 6 x 768

tab = tabella()
t = tab.search().where("escludibile=false").select(["url", "papa", "vector"]).limit(10**9).to_arrow()
urls = t.column("url").to_pylist(); papi = np.array(t.column("papa").to_pylist())
V = torch.from_numpy(np.stack(t.column("vector").to_pylist()).astype("float32"))
fam = (V @ A.T).argmax(1).numpy()    # la famiglia (ancora più vicina) di ogni chunk
print(len(urls), "chunk marcati")

In [ ]:
PAPI = ["Papa Giovanni Paolo II", "Papa Benedetto XVI", "Papa Francesco", "Papa Leone XIV"]
AB = ["GP2", "BXVI", "FRA", "LEO"]

print("composizione (% dei chunk) e lift per Papa\n")
print(f"{'famiglia':22}" + "".join(f"{a:>8}" for a in AB))
for fi, fn in enumerate(FN):
    perc = [100 * ((fam == fi) & (papi == p)).sum() / (papi == p).sum() for p in PAPI]
    media = float(np.mean(perc))
    lift = [x / media if media else 0 for x in perc]
    print(f"{fn:22}" + "".join(f"{perc[k]:5.1f}%" for k in range(4)))
    print(f"{'  lift':22}" + "".join(f"{lift[k]:6.2f} " for k in range(4)))

# quanto si mischiano i documenti
doc = collections.defaultdict(set)
for i, u in enumerate(urls):
    doc[u].add(fam[i])
nfam = [len(s) for s in doc.values()]
print(f"\nmix: {np.mean(nfam):.2f} famiglie/doc; "
      f"{100*np.mean([n >= 2 for n in nfam]):.0f}% dei doc ne tocca >=2, "
      f"{100*np.mean([n >= 3 for n in nfam]):.0f}% >=3")

## I numeri

**Composizione** (% dei chunk per famiglia) e **lift** (sopra/sotto la media dei quattro):

| famiglia | GP2 | BXVI | FRA | LEO | media |
|---|--:|--:|--:|--:|--:|
| liturgia | 3,4 (×1,13) | 3,7 (×1,23) | 1,7 (×0,57) | 2,3 (×0,77) | 3,0% |
| **fede e devozione** | 75,7 (×0,98) | 77,2 (×1,00) | 80,4 (×1,04) | 77,1 (×1,00) | **77,0%** |
| eventi e ricorrenze | 3,4 (×1,21) | 2,1 (×0,75) | 1,7 (×0,61) | 2,4 (×0,86) | 2,8% |
| viaggi | **9,9 (×1,15)** | 8,3 (×0,97) | 5,4 (×0,63) | 7,6 (×0,88) | 8,6% |
| programma e storia | 5,3 (×0,96) | 6,7 (×1,22) | 5,3 (×0,96) | 6,4 (×1,16) | 5,5% |
| **attualità** | 2,3 (×0,77) | 1,9 (×0,63) | **5,5 (×1,83)** | **4,3 (×1,43)** | 3,0% |

**Mix dei documenti:** in media **1,83 famiglie per documento**; il **56%** ne
tocca almeno due, il 21% almeno tre. I discorsi si mischiano davvero — liturgia
*e* altro nello stesso testo.

## Come si legge

**La linea rossa è ~l'80% di tutto, identica per tutti e quattro.** Fede/devozione
(77%) più liturgia (3%) fanno il grande continuo: lift attorno a 1 per ognuno. È
la **continuità**, vista dalla struttura — non un Papa diverso dagli altri, ma lo
stesso fondo per tutti.

**Le differenze vivono nella fascia sottile** (il 20% restante):
- **attualità** — Francesco ×1,83 e Leone ×1,43, contro GP2/Benedetto sotto media:
  è qui che si gioca l'accento "sociale" di cui parlano i giornali;
- **viaggi** — Giovanni Paolo II il più alto (×1,15), il grande viaggiatore;
- **programma e storia** — Benedetto (×1,22) e Leone (×1,16) un po' sopra;
- **liturgia/eventi** — più marcati nei due Papi "di curia" (GP2, Benedetto).

> ⚠️ **Caveat.** "Fede e devozione" è un attrattore enorme per il modello: quasi
> ogni pezzo di testo religioso gli cade vicino, perciò la lente delle famiglie è
> **grossa** — perfetta per il quadro macro (continuità + margini), meno per le
> distinzioni fini (per quelle, i temi puntuali del notebook 01 e i topic
> emergenti del 03).

---
*Solo viste **aggregate** (composizione e lift per famiglia), nessun testo
ripubblicato (© Libreria Editrice Vaticana, fonte `url` nei metadati).*